<a href="https://colab.research.google.com/github/mishrasubhajit/bengaliAsr/blob/main/MADASR_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!gsutil ls gs://prompt-injection_madasr

gs://prompt-injection_madasr/IISc_RESPIN_train_large/
gs://prompt-injection_madasr/IISc_RESPIN_train_small/


In [ ]:
!gsutil -m cp -r gs://prompt-injection_madasr/IISc_RESPIN_train_small/bn/D1/20050/content/bn_D1_20050

CommandException: Wrong number of arguments for "cp" command.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pytorch-lightning --quiet
!pip install wandb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.1/823.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.6/962.6 kB 61.8 MB/s eta 0:00:00


In [ ]:
!pip install jiwer pyctcdecode unidecode --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.6/518.6 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
!pip install https://github.com/kpu/kenlm/archive/master.zip

     \ 553.6 kB 4.5 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kenlm: filename=kenlm-0.2.0-cp311-cp311-linux_x86_64.whl size=3185034 sha256=bfb88e6e8d02e383674f0245acd70d5e48ca0ef23555d01c0d259636ffb2e2be
  Stored in directory: /tmp/pip-ephem-wheel-cache-fz9uu5uk/wheels/4e/ca/6a/e5da175b1396483f6f410cdb4cfe8bc8fa5e12088e91d60413
Successfully built kenlm


In [ ]:
!pip install flashlight-text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 14.5 MB/s eta 0:00:00


In [ ]:
import os
import time
import torch
import wandb
import torchaudio
import numpy as np
import torch.nn as nn
from jiwer import cer, wer
import pytorch_lightning as pl
import torch.nn.functional as F
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from torchaudio.models.decoder import ctc_decoder

## Build Vocabulary from Transcripts

In ASR (Automatic Speech Recognition), we need to convert characters in the transcript to numeric indices before feeding them into the model.

### This block:

	•	Reads all transcript files
	•	Counts all unique characters
	•	Builds a sorted vocabulary
### Adds special tokens:

	•	'-': CTC blank token (required by the loss function)
	•	'<pad>': used to pad variable-length target sequences
	•	'<unk>': for unknown characters
### Creates two mappings:

	•	char2idx: character → index
	•	idx2char: index → character

These mappings are used to:

	•	Convert transcripts into label tensors
	•	Decode model predictions back to readable text


In [ ]:
def build_vocab(transcript_paths):
    chars = Counter()

    # Step 1: Count all characters in all transcript files
    for path in transcript_paths:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                # Get the actual text part of the line (after ID)
                text = line.strip().split(' ', 1)[-1]
                # Update character frequency
                chars.update(text)

    # Step 2: Get unique sorted characters from the dataset
    vocab = sorted(set(chars))

    # Step 3: Add special tokens
    vocab = ['-', '<pad>', '<unk>'] + vocab  # '-' = blank (CTC), <pad> = padding, <unk> = unknown

    # Step 4: Create character-to-index and index-to-character maps
    char2idx = {c: i for i, c in enumerate(vocab)}
    idx2char = {i: c for c, i in char2idx.items()}

    return char2idx, idx2char, vocab

## 📚 RESPINASRDataset: Loading and Preprocessing Audio for ASR

### This dataset class:

	•	Recursively reads all .wav files and transcripts (.txt) from a structured RESPIN dataset folder
	•	Converts .wav audio into Mel spectrograms using GPU (if available)
	•	Converts the corresponding text transcript into a sequence of character indices using char2idx
	•	Caches computed Mel spectrograms to disk (in /content/mel_cache/) to avoid recomputing on every epoch

### This class supports:

	•	Efficient GPU-based preprocessing
	•	Caching with PyTorch .pt files
	•	Easy use with PyTorch DataLoader for batching

In [ ]:
class RESPINASRDataset(Dataset):
    def __init__(self, root_dir, char2idx, cache_dir="/content/mel_cache", sample_rate=16000, n_mels=80):
        self.data = []  # Stores (wav_path, base_id, text) triples
        self.char2idx = char2idx
        self.cache_dir = cache_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Create the Mel spectrogram transform (log-mel filterbank)
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=1024,
            hop_length=256,
            n_mels=n_mels,
            center=False
        ).to(self.device)

        # Convert raw spectrogram to decibels
        self.db_transform = torchaudio.transforms.AmplitudeToDB().to(self.device)

        # Create the cache directory if it doesn’t exist
        os.makedirs(self.cache_dir, exist_ok=True)

        # Walk through the dataset structure
        for domain in sorted(os.listdir(root_dir)):
            domain_path = os.path.join(root_dir, domain)
            if not os.path.isdir(domain_path): continue

            for folder in sorted(os.listdir(domain_path)):
                folder_path = os.path.join(domain_path, folder)
                if not os.path.isdir(folder_path): continue

                # Each subfolder should contain one transcript .txt file
                txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
                if not txt_files:
                    continue

                txt_path = os.path.join(folder_path, txt_files[0])

                # Build a set of available .wav files (base name only)
                available_wavs = {os.path.splitext(f)[0] for f in os.listdir(folder_path) if f.endswith('.wav')}

                # Read the transcript file and pair with corresponding audio
                with open(txt_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        parts = line.strip().split('\t', 1)
                        if len(parts) != 2:
                            continue
                        base, text = parts[0].strip(), parts[1].strip()
                        if base in available_wavs:
                            wav_path = os.path.join(folder_path, base + ".wav")
                            self.data.append((wav_path, base, text))

        print(f"✅ Loaded {len(self.data)} wav-text pairs.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        wav_path, base_id, text = self.data[idx]
        cache_path = os.path.join(self.cache_dir, base_id + ".pt")

        # Try to load precomputed mel spectrogram from disk
        if os.path.exists(cache_path):
            mel_spec_db = torch.load(cache_path)
        else:
            # Load raw waveform from disk
            waveform, _ = torchaudio.load(wav_path)

            # Convert stereo to mono if needed
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)

            # Move waveform to GPU if available
            waveform = waveform.to(self.device)

            # Compute Mel spectrogram and convert to decibels
            mel_spec = self.mel_transform(waveform)
            mel_spec_db = self.db_transform(mel_spec).squeeze(0).cpu()  # save to CPU

            # Cache the mel spectrogram
            torch.save(mel_spec_db, cache_path)

        # Convert transcript to character-level tensor
        target = [self.char2idx.get(c, self.char2idx['<unk>']) for c in text]

        return mel_spec_db, torch.tensor(target), text

In [ ]:
# Step 1: build vocabulary
import glob
transcript_paths = glob.glob("/content/drive/MyDrive/madasr/IISc_RESPIN_train_small/bn/D*/20*/IISc_RESPIN_*.txt")
char2idx, idx2char, vocab = build_vocab(transcript_paths)

# Step 2: load dataset
dataset = RESPINASRDataset("/content/drive/MyDrive/madasr/IISc_RESPIN_train_small/bn", char2idx)

# Step 3: inspect a sample
mel, target, raw_text = dataset[0]
print("Mel shape:", mel.shape)
print("Target:", target)
print("Text:", raw_text)

✅ Loaded 17160 wav-text pairs.
Mel shape: torch.Size([80, 459])
Target: tensor([31, 58, 41, 53, 29,  3, 19, 52, 45, 62, 31,  3, 50, 34, 53, 25, 58,  3,
        21, 29, 58,  3, 45, 19, 43, 58, 45,  3, 19, 52, 45, 62, 31,  3, 44, 58,
        29, 52,  3, 35, 58, 19, 58,  3, 49, 23, 62, 21, 58,  3, 49, 23, 62, 21,
        58,  3, 29, 52, 19, 52,  3, 19, 52, 29, 52,  3, 44, 52, 66])
Text: ডেবিট কার্ড হতিছে গটে রকমের কার্ড যেটা থেকে সঙ্গে সঙ্গে টাকা কাটা যায়


## 🧩 collate_fn: Custom Batching for ASR Inputs

In ASR tasks, each audio file and transcript can be a different length. PyTorch’s default DataLoader can’t batch variable-length inputs directly.

### This function:

	•	Accepts a list of items from your dataset: (mel_spec, target_tensor, raw_text)
	•	Pads all Mel spectrograms to the same time length
	•	Pads target sequences (transcripts) to the same length
	•	Returns input/target lengths for use with CTC loss

This makes sure your model can process batches without worrying about variable shapes.


In [ ]:
def collate_fn(batch):
    """
    Collates a batch of variable-length Mel spectrograms and transcripts into a uniform batch.

    Args:
        batch: list of tuples (mel_spec: [n_mels, T], target_tensor: [L], raw_text: str)

    Returns:
        padded_specs: Tensor of shape [B, n_mels, max_T] — padded Mel spectrograms
        input_lengths: Tensor [B] — original length (in frames) of each mel spec
        padded_targets: Tensor [B, max_L] — padded character index sequences
        target_lengths: Tensor [B] — original transcript lengths
        texts: list of original transcript strings
    """

    # Step 1: Transpose Mel spectrograms to [T, n_mels] so they can be padded on time axis
    specs = [item[0].transpose(0, 1) for item in batch]  # now [T, n_mels] per sample

    # Step 2: Collect target sequences and raw text
    targets = [item[1] for item in batch]
    texts = [item[2] for item in batch]

    # Step 3: Pad spectrograms by time axis (dim=0 after transpose)
    padded_specs = pad_sequence(specs, batch_first=True)  # [B, max_T, n_mels]
    padded_specs = padded_specs.transpose(1, 2)  # [B, n_mels, max_T] → required by model

    # Step 4: Pad target sequences (char indices)
    padded_targets = pad_sequence(targets, batch_first=True, padding_value=0)

    # Step 5: Get original lengths (for use with CTC loss)
    input_lengths = torch.tensor([s.shape[0] for s in specs], dtype=torch.int32)  # time frames per input
    target_lengths = torch.tensor([t.size(0) for t in targets], dtype=torch.int32)

    return padded_specs, input_lengths, padded_targets, target_lengths, texts

In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,       # Try 2 or 4 for Colab/VMs; avoid 0 unless debugging
    pin_memory=False
)

# Example batch
batch = next(iter(dataloader))
padded_specs, input_lengths, padded_targets, target_lengths, raw_texts = batch

print("Padded specs:", padded_specs.shape)
print("Input lengths:", input_lengths)
print("Padded targets:", padded_targets.shape)
print("Target lengths:", target_lengths)
print("First text:", raw_texts[0])

Padded specs: torch.Size([4, 80, 523])
Input lengths: tensor([340, 523, 368, 260], dtype=torch.int32)
Padded targets: torch.Size([4, 58])
Target lengths: tensor([56, 57, 58, 41], dtype=torch.int32)
First text: ভেড়া যে খামারে থাকছে সেখানে অনেক রকমের আয়োজন রাখতে হচ্ছে


## 🎙️ LitASRModel: A Deep Learning Model for Speech-to-Text

This is a PyTorch Lightning model for automatic speech recognition (ASR) using:

	•	🧠 CNN to extract local acoustic features from Mel spectrograms
	•	🔁 Bidirectional LSTM (BiLSTM) to capture long-range temporal dependencies
	•	🎯 CTC loss (Connectionist Temporal Classification) to align audio and text without explicit alignment

### 🧪 Evaluation Metrics

	•	CER (Character Error Rate): Measures how many characters are wrong in the prediction. Lower = better (0% = perfect).
	•	WER (Word Error Rate): Similar to CER but at the word level.

### 🧮 CTC Loss: Why We Use It

Traditional sequence models require aligned input/output sequences (e.g., frame-by-frame).
But in speech, we often don’t know which frame corresponds to which character.

🧠 CTC allows the model to learn alignments implicitly, by predicting blank tokens ('-') between and around real tokens.

#### In ASR (speech-to-text), we have:
	•	An input sequence: acoustic features over time → X = [x₁, x₂, ..., x_T]
	•	A target sequence: text characters → Y = [y₁, y₂, ..., y_L]

But:

	•	The input is much longer (T ≫ L)
	•	We don’t know the alignment between input frames and output characters

CTC loss solves this by:

	•	Allowing the model to emit blank tokens (-)
	•	Allowing repeated characters (e.g., h-e-ll-o maps to hello)
	•	Marginalizing over all valid alignments

⸻

🧮 CTC Loss — High-Level Formula

Let:
	•	π = [π₁, π₂, ..., π_T] be a possible path (a sequence of tokens at every timestep)
	•	B(π) be the function that removes repeated characters and blanks
	•	Example: B([a, -, a, a, -, b]) = aab

Then CTC loss minimizes the negative log probability of all valid paths:

$
\mathcal{L}_{\text{CTC}} = - \log \sum{\pi \in \mathcal{A}(Y)} P(\pi | X)
$

Where:

- $\mathcal{A}(Y)$ is the set of all valid alignments (paths π that reduce to Y)
- $P(\pi \mid X)$ is the probability of alignment π given the input

### 🔀 Decoding Strategies

	•	Greedy Decoding: Pick the most likely token at each time step. Fast but less accurate.
	•	Beam Search: Keeps multiple hypotheses at each step. More accurate, but slower.
	•	This model currently uses greedy decoding for speed and debugging.

In [ ]:
class LitASRModel(pl.LightningModule):
    def __init__(self, vocab, char2idx, idx2char, lr=1e-3, input_dim=80, cnn_channels=64, lstm_dim=256, num_lstm_layers=3):
        super().__init__()

        # Save hyperparameters
        self.save_hyperparameters(ignore=["vocab", "char2idx", "idx2char"])
        self.vocab = vocab
        self.char2idx = char2idx
        self.idx2char = idx2char
        self.blank_idx = char2idx["-"]  # CTC blank token
        self.lr = lr

        # Optional CTC beam decoder (not used in training here)
        self.decoder = ctc_decoder(
            lexicon=None,
            tokens=vocab,
            lm=None,
            blank_token='-',
            sil_token='<pad>',
            unk_word='<unk>'
        )

        # CNN encoder for time-frequency patterns
        self.cnn = nn.Sequential(
            nn.Conv2d(1, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_channels),
            nn.ReLU(),
            nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_channels),
            nn.ReLU(),
        )

        # Bidirectional LSTM for sequence modeling
        self.lstm = nn.LSTM(
            input_size=cnn_channels * input_dim,
            hidden_size=lstm_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True,
        )

        # Final classifier (logits over vocab)
        self.classifier = nn.Linear(lstm_dim * 2, len(vocab))

        # CTC loss function (zero_infinity avoids NaN errors)
        self.ctc_loss = nn.CTCLoss(blank=self.blank_idx, zero_infinity=True)

    def forward(self, x, input_lengths):
        # x: [B, n_mels, T] → [B, 1, n_mels, T]
        x = x.unsqueeze(1)
        x = self.cnn(x)  # [B, C, F, T]
        b, c, f, t = x.size()
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)  # [B, T, C×F]
        x, _ = self.lstm(x)  # [B, T, 2*hidden]
        x = self.classifier(x)  # [B, T, vocab_size]
        return x

    def greedy_decode(self, log_probs, input_lengths):
        # Greedy decoding: take argmax at each frame, remove repeats and blanks
        pred_ids = log_probs.argmax(-1)
        decoded_texts = []
        for pred in pred_ids:
            tokens = []
            prev = self.blank_idx
            for idx in pred:
                idx = idx.item()
                if idx != prev and idx != self.blank_idx:
                    tokens.append(self.idx2char.get(idx, ""))
                prev = idx
            decoded_texts.append("".join(tokens))
        return decoded_texts

    def compute_metrics(self, log_probs, input_lengths, raw_texts):
        decoded_texts = self.greedy_decode(log_probs, input_lengths)
        cer_scores = [cer(gt, pred) for gt, pred in zip(raw_texts, decoded_texts)]
        wer_scores = [wer(gt, pred) for gt, pred in zip(raw_texts, decoded_texts)]
        avg_cer = 100 * sum(cer_scores) / len(cer_scores)
        avg_wer = 100 * sum(wer_scores) / len(wer_scores)
        return avg_cer, avg_wer, decoded_texts

    def training_step(self, batch, batch_idx):
        specs, input_lens, targets, target_lens, _ = batch
        # Move to correct device
        specs = specs.to(self.device)
        input_lens = input_lens.to(self.device)
        targets = targets.to(self.device)
        target_lens = target_lens.to(self.device)

        # Forward + CTC loss
        logits = self(specs, input_lens)
        log_probs = F.log_softmax(logits, dim=-1)
        loss = self.ctc_loss(log_probs.transpose(0, 1), targets, input_lens, target_lens)

        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        specs, input_lens, targets, target_lens, raw_texts = batch
        specs = specs.to(self.device)
        input_lens = input_lens.to(self.device)
        targets = targets.to(self.device)
        target_lens = target_lens.to(self.device)

        logits = self(specs, input_lens)
        log_probs = F.log_softmax(logits, dim=-1)
        loss = self.ctc_loss(log_probs.transpose(0, 1), targets, input_lens, target_lens)
        self.log("val_loss", loss, prog_bar=True)

        cer_score, wer_score, decoded_texts = self.compute_metrics(log_probs, input_lens, raw_texts)
        self.log("val_cer", cer_score, prog_bar=True)
        self.log("val_wer", wer_score, prog_bar=True)

        if batch_idx == 0:
            print("🔊 Predicted:", decoded_texts[0])
            print("📜 Ground Truth:", raw_texts[0])

        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr)
        scheduler = {
            "scheduler": torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", factor=0.5, patience=2, verbose=True
            ),
            "monitor": "val_cer"
        }
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

## 🚀 Training Setup with PyTorch Lightning

This block sets up the full training loop using PyTorch Lightning.

### ✅ What It Does:
1. **Splits the dataset** into training and validation sets
2. **Initializes DataLoaders** for batching, padding, and shuffling
3. **Instantiates the ASR model**
4. **Configures WandB** for experiment tracking
5. **Adds training callbacks**:
   - ⏳ **EarlyStopping**: stops if no improvement in validation CER
   - 💾 **ModelCheckpoint**: saves the best-performing model
   - 📉 **LearningRateMonitor**: logs LR for dynamic schedulers
6. **Creates a Lightning Trainer**
7. **Starts training** with `.fit(...)`


In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from torch.utils.data import random_split, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

wandb.finish()
# 1. Create DataLoaders
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn, num_workers=0,        # or 4, depending on CPU
    pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0,        # or 4, depending on CPU
    pin_memory=False)


# 3. Initialize model
model = LitASRModel(
    vocab=vocab,
    char2idx=char2idx,
    idx2char=idx2char,
    lr=1e-4,
    input_dim=80
)

# 4. WandB Logger
wandb_logger = WandbLogger(project="madasr-asr", name="bengali-track1")

# 5. Callbacks
checkpoint_callback = ModelCheckpoint(
    monitor="val_cer",
    mode="min",
    save_top_k=1,
    filename="best-val-cer-{epoch:02d}-{val_cer:.2f}",
    verbose=True
)

early_stop_callback = EarlyStopping(
    monitor="val_cer",
    patience=5,
    verbose=True,
    mode="min"
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

# 6. Trainer
trainer = Trainer(
    logger=wandb_logger,
    max_epochs=40,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    log_every_n_steps=40,
    num_sanity_val_steps=1,
    callbacks=[checkpoint_callback, early_stop_callback, lr_monitor]
)

# 7. Start training
trainer.fit(model, train_loader, val_loader)

epoch,▁▁▁▁▁▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇██████▁
lr-AdamW,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,██████▇▆▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁█
trainer/global_step,▁▁▁▁▁▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█▁▁▁▁
val_cer,██▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁█
val_loss,█▇▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁█
val_wer,███▇▅▅▄▃▃▃▂▂▂▁▁▁▁▁▁▁█
epoch,1
lr-AdamW,0.0001
train_loss,3.28304
trainer/global_step,1159


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name       | Type       | Params | Mode 
--------------------------------------------------
0 | cnn        | Sequential | 37.8 K | train
1 | lstm       | LSTM       | 14.2 M | train
2 | classifier | Linear     | 34.4 K | train
3 | ctc_loss   | CTCLoss    | 0      | train
--------------------------------------------------
14.2 M    Trainable params
0         Non-trainable params
14.2 M    Total params
56.961    Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

🔊 Predicted: থ<unk> ম মদথমথমধমজমঝমঝগথমফ জমআত ম ঙ মঝম মথ ম
📜 Ground Truth: কানাডার পঁচাত্তর শতাংশ মানষি শহরত বসবাস করে


...

🔊 Predicted: কানাডাল প্রচারতু শোতাহশ মানসি সরত পশবাস করে
📜 Ground Truth: কানাডার পঁচাত্তর শতাংশ মানষি শহরত বসবাস করে


INFO:pytorch_lightning.utilities.rank_zero:Epoch 26, global step 26082: 'val_cer' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

🔊 Predicted: কানাডাল প্রচারতু শোতাশ মানসি সগরত পশবাস করে
📜 Ground Truth: কানাডার পঁচাত্তর শতাংশ মানষি শহরত বসবাস করে


INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_cer did not improve in the last 5 records. Best score: 24.473. Signaling Trainer to stop.
INFO:pytorch_lightning.utilities.rank_zero:Epoch 27, global step 27048: 'val_cer' was not in top 1


### Pure Pytorch code

In [ ]:
class ASRModel(nn.Module):
    def __init__(self, vocab, char2idx, idx2char, input_dim=80, cnn_channels=64, lstm_dim=256, num_lstm_layers=3):
        super().__init__()
        self.vocab = vocab
        self.char2idx = char2idx
        self.idx2char = idx2char
        self.blank_idx = char2idx["-"]

        self.cnn = nn.Sequential(
            nn.Conv2d(1, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_channels),
            nn.ReLU(),
            nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_channels),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(
            input_size=cnn_channels * input_dim,
            hidden_size=lstm_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True,
        )

        self.classifier = nn.Linear(lstm_dim * 2, len(vocab))
        self.ctc_loss = nn.CTCLoss(blank=self.blank_idx, zero_infinity=True)

    def forward(self, x, input_lengths):
        x = x.unsqueeze(1)  # [B, 1, n_mels, T]
        x = self.cnn(x)     # [B, C, F, T]
        b, c, f, t = x.size()
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)  # [B, T, C×F]
        x, _ = self.lstm(x)
        x = self.classifier(x)  # [B, T, vocab_size]
        return x

    def compute_loss(self, specs, input_lens, targets, target_lens):
        logits = self(specs, input_lens)
        log_probs = F.log_softmax(logits, dim=-1)
        loss = self.ctc_loss(log_probs.transpose(0, 1), targets, input_lens, target_lens)
        return loss, log_probs

    def greedy_decode(self, log_probs, input_lengths):
        pred_ids = log_probs.argmax(-1)
        decoded_texts = []
        for pred in pred_ids:
            tokens = []
            prev = self.blank_idx
            for idx in pred:
                idx = idx.item()
                if idx != prev and idx != self.blank_idx:
                    tokens.append(self.idx2char.get(idx, ""))
                prev = idx
            decoded_texts.append("".join(tokens))
        return decoded_texts

    def compute_metrics(self, log_probs, input_lengths, raw_texts):
        decoded_texts = self.greedy_decode(log_probs, input_lengths)
        cer_scores = [cer(gt, pred) for gt, pred in zip(raw_texts, decoded_texts)]
        wer_scores = [wer(gt, pred) for gt, pred in zip(raw_texts, decoded_texts)]
        avg_cer = 100 * sum(cer_scores) / len(cer_scores)
        avg_wer = 100 * sum(wer_scores) / len(wer_scores)
        return avg_cer, avg_wer, decoded_texts

In [ ]:
def train_and_validate(model, train_loader, val_loader, optimizer, num_epochs, device="cuda"):
    model = model.to(device)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

    wandb.init(project="madasr-asr", name="bengali-track1-pytorch", reinit=True)
    best_val_cer = float("inf")

    for epoch in range(num_epochs):
        print(f"\n🔁 Epoch {epoch + 1}/{num_epochs}")

        # Training phase
        model.train()
        train_losses = []
        for batch in tqdm(train_loader, desc="Training"):
            specs, input_lens, targets, target_lens, _ = batch
            specs = specs.to(device)
            input_lens = input_lens.to(device)
            targets = targets.to(device)
            target_lens = target_lens.to(device)

            optimizer.zero_grad()
            loss, _ = model.compute_loss(specs, input_lens, targets, target_lens)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        avg_train_loss = sum(train_losses) / len(train_losses)
        wandb.log({"train_loss": avg_train_loss}, step=epoch)
        print(f"✅ Train Loss: {avg_train_loss:.4f}")

        # Validation phase
        model.eval()
        val_losses = []
        val_cers = []
        val_wers = []

        with torch.no_grad():
            for i, batch in enumerate(tqdm(val_loader, desc="Validation")):
                specs, input_lens, targets, target_lens, raw_texts = batch
                specs = specs.to(device)
                input_lens = input_lens.to(device)
                targets = targets.to(device)
                target_lens = target_lens.to(device)

                loss, log_probs = model.compute_loss(specs, input_lens, targets, target_lens)
                val_losses.append(loss.item())

                cer_score, wer_score, decoded = model.compute_metrics(log_probs, input_lens, raw_texts)
                val_cers.append(cer_score)
                val_wers.append(wer_score)

                if i == 0:
                    print("🔊 Prediction:", decoded[0])
                    print("📜 Reference :", raw_texts[0])

        avg_val_loss = sum(val_losses) / len(val_losses)
        avg_val_cer = sum(val_cers) / len(val_cers)
        avg_val_wer = sum(val_wers) / len(val_wers)

        wandb.log({
            "val_loss": avg_val_loss,
            "val_cer": avg_val_cer,
            "val_wer": avg_val_wer,
            "epoch": epoch
        }, step=epoch)

        print(f"📉 Val Loss: {avg_val_loss:.4f} | CER: {avg_val_cer:.2f}% | WER: {avg_val_wer:.2f}%")

        scheduler.step(avg_val_cer)

        # Save best model
        if avg_val_cer < best_val_cer:
            best_val_cer = avg_val_cer
            ckpt_path = f"best_model_epoch{epoch+1}_cer{avg_val_cer:.2f}.pt"
            torch.save(model.state_dict(), ckpt_path)
            print(f"💾 Saved new best model to {ckpt_path}")

    wandb.finish()

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

train_and_validate(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_epochs=40,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

Characer error rate (CER) = 25; Word error rate (WER) = 68